In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# OpenAI Chat Completions API 기본
이 튜토리얼은 OpenAI의 Chat Completions API를 활용하여 챗봇이나 AI 기능을 개발하는 방법을
단계별로 설명합니다. 특히 OpenAI의 최신 언어 모델 중 하나인 GPT-4o-mini/gpt-4.1-nano를 사
용하여 예제를 진행할 것입니다. 각 섹션에는 개념 설명과 함께 실행 가능한 파이썬 코드 예제가
포함되어 있습니다.

### 주요 학습 내용:
1. OpenAI API 소개 및 환경 설정: OpenAI API 개요, API 키 발급 및 보안 설정, 파이썬 클라이언트 설치 및 인스턴스 생성 방법
2. 기본적인 Chat Completions API 사용법: 간단한 대화형 텍스트 생성 요청과 응답 처리, 프롬프트 엔지니어링 기초
3. 스트리밍 응답: 대화 응답을 스트리밍 방식으로 받아 실시간 처리하는 방법
4. 시스템 메시지 활용: 시스템 역할 메시지를 사용하여 AI의 응답 스타일이나 행동을 조정하는 방법
5. 고급 활용법: 토큰 최적화와 비용 절감 전략, OpenAI API 에러 처리 및 예외Handling
6. 실전 프로젝트 예제: 간단한 챗봇 구현 및 외부 데이터/API와 연동하여 데이터 분석 기능을 결합한 사례

## 1. OpenAI API 소개 및 환경 설정
먼저 OpenAI API와 Chat Completions에 대해 간략히 알아보고, API를 사용하기 위한 환경을 설정
해보겠습니다.

### OpenAI API 개요
OpenAI API는 GPT 계열의 대규모 언어 모델을 인터넷을 통해 사용할 수 있도록 제공하는 서비스
입니다. Chat Completions API는 챗봇과 유사한 대화형 상호작용을 할 수 있는 엔드포인트로, 역할
(role)이 부여된 메시지 목록을 입력하면 모델이 다음 대화 내용을 생성합니다. GPT-4o는 텍스트와 이미지 입력을 모두 처리하며 최대 128k 토큰의 긴 문맥을 다룰 수 있습니다. GPT-4o와 경량화 모델인 GPT-4o-mini 등이 제공되며, 요구 사항에 따라 적절한 모델을 선택할 수 있습니다
(GPT-4o-mini는 비용 효율이 높음)

### API 키 발급 및 보안 설정
OpenAI API를 사용하려면 먼저 OpenAI 계정에서 API 키를 발급받아야 합니다. OpenAI 웹사이트
의 API Keys 페이지에서 새로운 비밀 키를 생성할 수 있습니다. 발급받은 API 키는 비밀로 관리해
야 하며, 소스 코드나 공개 저장소에 노출되지 않도록 주의해야 합니다. 가장 좋은 방법은 API 키
를 코드에 하드코딩하지 않고, 환경 변수나 별도의 설정 파일에 저장하는 것입니다. 이 튜토리얼
에서는 .env 파일에 키를 저장하고 파이썬에서 이를 불러오는 방식을 사용합니다. 이를 위해
Python용 패키지 **python-dotenv**를 활용하겠습니다.
- .env 파일에 키 저장: 프로젝트 디렉터리에 .env 파일을 만들고 아래와 같이 API 키를 저장합니
다 (따옴표 없이).
 ```
 OPENAI_API_KEY=발급받은-API키-값
 ```
- python-dotenv 사용: 파이썬 코드에서 python-dotenv를 이용해 .env 파일의 환경 변수를 불러
올 수 있습니다.


In [9]:
import openai
openai.__version__
# 설정 -> 개인정보 및 보안 -> 앱 및 브라우저컨트롤 -> 스마트앱컨트롤 끄기

# openAㅑ 

'3.11.0'

In [12]:
from dotenv import load_dotenv
load_dotenv(
    # dotenv_path='e:/.env'
    ) # .env파일의 key와 값을 시스템 환경변수로 셋팅
import os
os.getenv('OPENAI_API_KEY')[:3]

'sk-'

In [13]:
from openai import OpenAI
client = OpenAI(
            #api_key=os.getenv('OPENAI_API_KEY')
)

In [33]:
# 설정 -> 개인정보 및 보안 -> 앱 및 브라우저컨트롤 -> 스마트앱컨트롤 끄기
response = client.responses.create(
    model="gpt-4o-mini", 
    input="Tell me a funny joke"
)
print(response.output_text)

Why don't scientists trust atoms? 

Because they make up everything!


In [34]:
response.output

[ResponseOutputMessage(id='msg_0580844dce28647c006aa7592bbabc87d08505b3d419ab760d', content=[ResponseOutputText(annotations=[], text="Why don't scientists trust atoms? \n\nBecause they make up everything!", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]

In [18]:
print(response.output[0].content[0].text)

Why did the scarecrow win an award?

Because he was outstanding in his field!


In [20]:
response.usage

ResponseUsage(input_tokens=12, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=18, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=30)

In [21]:
response = client.responses.create(
    model="gpt-4o-mini", 
    input="웃긴 농담하나 해줘"
)
print(response.output_text)

물고기가 학교에 가면 어떤 과목을 배우는지 알아? 

"물리!" 🎣😄


In [22]:
print(response.output[0].content[0].text)

물고기가 학교에 가면 어떤 과목을 배우는지 알아? 

"물리!" 🎣😄


In [23]:
response.usage

ResponseUsage(input_tokens=15, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=26, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=41)

In [36]:
# 추론 모델
response = client.responses.create(
    model='gpt-5-nano',
    input='웃긴 농담 하나 해줘'
)
print(response.output_text)

자전거는 왜 항상 혼자 있을 수 없을까요? 항상 두 바퀴가 있어야 하니까요. 
원하시면 분위기나 스타일을 바꾼 또 다른 농담도 drôle 드릴게요!


In [44]:
response.output[0] # 추론 과정의 데이터
response.output[1] # message (실제 output 결과)

# 추론모델 : o1, o3, o3-mini, o4-mini(o시리즈), gpt-5, gpt-5-mini, gpt-5-nano..(gpt5시리스)
# 추론모델이 아닌 모델 : gpt-4o-mini, gpt-4.1, (gpt-4이하의 모델)
response.output[1].content[0].text

'자전거는 왜 항상 혼자 있을 수 없을까요? 항상 두 바퀴가 있어야 하니까요. \n원하시면 분위기나 스타일을 바꾼 또 다른 농담도 drôle 드릴게요!'

In [51]:
# response.output_text는 @property로 작성된 함수
class Person:
    def __init__(self, name):
        self.name = name
    @property
    def output(self):
        return '결과'
p = Person("홍길동")
print(p.name)
print(p.output)

홍길동
결과


위 코드로 client 객체가 생성되었습니다. 이제 이 client를 통해 OpenAI Chat Completions API를
호출할 수 있습니다. 다음 섹션부터는 실제로 Chat Completions API를 호출하여 다양한 기능을 실
습해보겠습니다.

## 2. 기본적인 Chat Completions API 사용법
이 섹션에서는 Chat Completions API를 사용하여 가장 기본적인 대화 생성 작업을 수행해봅니다.

### 간단한 텍스트 생성 요청
Chat Completions 엔드포인트는 메시지 목록을 입력으로 받아 다음에 이어질 메시지를 생성합니다. 각 메시지는 role과 content 필드로 구성되어 있으며, 일반적으로 **user (사용자 메시지), assistant (모델의 응답 메시지), system (시스템 지시 메시지)** 세 가지 역할을 사용합니다. 가장 간단한 예제로, 사용자 역할의 메시지 하나를 모델에 보내고 응답을 받아보겠습니다. 모델은 gpt-4.1-nano를 사용합니다.

In [52]:
# 사용자 메세지 구성
messages = [
    {'role':'user', 'content':'안녕하세요. 오늘 날씨가 어떤가요?'}
]
response = client.chat.completions.create(
    model = 'gpt-4.1-nano', # 추론모델이 아닌 모델
    messages=messages,
    temperature=0.7, # 0~2 : 일관적 ~ 창의적
    frequency_penalty=0.5, # 빈도 보정 -2~2 : 값이 클수록 단어/토큰 반복 억제
)
response

ChatCompletion(id='chatcmpl-ENr7h0SWPHFnKFrwNmLK9ecxjgBLH', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 죄송하지만, 저는 실시간 날씨 정보를 제공할 수 없습니다. 오늘의 정확한 날씨를 확인하시려면 날씨 앱이나 기상청 웹사이트를 참고하시길 추천드립니다. 도움이 필요하시면 다른 질문도 환영합니다!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789354457, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_1c77073857', usage=CompletionUsage(completion_tokens=57, prompt_tokens=18, total_tokens=75, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [55]:
response.choices[0].message.content

'안녕하세요! 죄송하지만, 저는 실시간 날씨 정보를 제공할 수 없습니다. 오늘의 정확한 날씨를 확인하시려면 날씨 앱이나 기상청 웹사이트를 참고하시길 추천드립니다. 도움이 필요하시면 다른 질문도 환영합니다!'

In [59]:
messages = [
    {'role':'user', 'content':'안녕하세요. 오늘 서울 날씨가 어떤가요?'}
]
response = client.chat.completions.create(
    model = 'gpt-5-nano', # 추론모델
    messages=messages,
    # temperature=0.7, # 0~2 : 일관적 ~ 창의적
    # frequency_penalty=0.5, # 빈도 보정 -2~2 : 값이 클수록 단어/토큰 반복 억제
    reasoning_effort='minimal' # minimal/low/medium/high(깊게 추론할수록 output token이 많이 소요)
)
response

ChatCompletion(id='chatcmpl-ENrEoToLrUkGPinE4KvZCEUo4zJyS', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='죄송하지만 지금 실시간 날씨 정보에 접근할 수 없습니다.\n\n서울의 최신 날씨를 확인하려면 다음 방법을 이용해 보세요.\n- 기상청(KMA) 또는 기상정보 사이트에서 “오늘의 서울 날씨” 확인\n- 네이버 날씨, 다음 날씨 등 포털 날씨 서비스\n- 날씨 앱(Apple Weather, Google Weather, Weather.com 등)\n\n원하시면 제가 today 기준의 일반적인 계절별 특징이나 옷차림 팁은 드릴 수 있고, 특정 시간대의 날씨를 예측해 주는 웹사이트 이용 방법도 안내해 드릴게요. 어떤 방식으로 도와드릴까요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789354898, model='gpt-5-nano-2025-08-07', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=281, prompt_tokens=18, total_tokens=299, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=128, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTok

In [60]:
response.choices[0].message.content

'죄송하지만 지금 실시간 날씨 정보에 접근할 수 없습니다.\n\n서울의 최신 날씨를 확인하려면 다음 방법을 이용해 보세요.\n- 기상청(KMA) 또는 기상정보 사이트에서 “오늘의 서울 날씨” 확인\n- 네이버 날씨, 다음 날씨 등 포털 날씨 서비스\n- 날씨 앱(Apple Weather, Google Weather, Weather.com 등)\n\n원하시면 제가 today 기준의 일반적인 계절별 특징이나 옷차림 팁은 드릴 수 있고, 특정 시간대의 날씨를 예측해 주는 웹사이트 이용 방법도 안내해 드릴게요. 어떤 방식으로 도와드릴까요?'

In [61]:
# few shot
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하게 답변해주는 비서야'}, # 역할 부여
        {'role':'user',   'content':'프랑스 수도는?'}, # few shot(모범 답안)
        {'role':'assistant', 'content':'파리(수도명만 대답)'},
        {'role':'user',   'content':'이탈리아 수도는?'}, # few shot(모범 답안)
        {'role':'assistant', 'content':'로마(수도명만 대답)'},
        {'role':'user',   'content':'한국 수도는?'}
    ],
)
response

ChatCompletion(id='chatcmpl-ENrOSU5VWT9yKZDFVEMOqiEpB8Y3X', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='서울', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789355496, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_cbf9666c94', usage=CompletionUsage(completion_tokens=1, prompt_tokens=75, total_tokens=76, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [62]:
response.choices[0].message.content

'서울'

In [65]:
# 역할 설정
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 컴퓨터 프로그램 전문가야'}, # 역할 부여
        {'role':'user',   'content':'Spring이 뭐야?'} # 질문
    ],
    temperature = 1,
    frequency_penalty=0.5
)
print(response.choices[0].message.content)

Spring은 자바 기반의 오픈 소스 프레임워크로, 엔터프라이즈 애플리케이션 개발을 간소화하고 효율적으로 만들기 위해 설계되었습니다. 주요 특징과 기능은 다음과 같습니다:

1. **경량화된 컨테이너 (Inversion of Control, IoC)**: Spring은 의존성 주입(Dependency Injection)을 통해 객체 간의 결합도를 낮추고, 쉽게 테스트하고 유지보수할 수 있는 구조를 제공합니다.

2. **모듈화된 아키텍처**: Spring은 여러 모듈로 구성되어 있어, 필요에 따라 선택적으로 사용할 수 있습니다. 대표적인 모듈로는 Spring Core, Spring MVC, Spring Data, Spring Security 등이 있습니다.

3. **웹 개발 지원**: Spring MVC를 통해 RESTful API와 웹 애플리케이션을 신속하게 개발할 수 있습니다.

4. **트랜잭션 관리 및 데이터 액세스 지원**: 데이터베이스 연결과 트랜잭션 처리를 쉽게 할 수 있도록 도와줍니다.

5. **유연성과 확장성**: 다양한 기술과 연동이 용이하여, 기존 시스템과의 통합도 원활히 할 수 있습니다.

간단히 말해, Spring은 자바 개발자가 복잡한 엔터프라이즈 애플리케이션을 손쉽게 설계하고 구현할 수 있도록 도와주는 강력한 프레임워크입니다.


In [66]:
# 역할 설정
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 문학 전문가야. 특히 시를 좋아해'}, # 역할 부여
        {'role':'user',   'content':'Spring이 뭐야?'} # 질문
    ],
    temperature = 1,
    frequency_penalty=0.5
)
print(response.choices[0].message.content)

봄(Spring)은 사계절 가운데 한 계절로, 겨울 끝에 시작되어 여름이 오기 전까지 지속되는 따뜻하고 생기가 넘치는 시기를 의미해요. 자연은 푸른 새싹과 꽃이 피어나고, 나무는 다시 잎을 풍성하게 돋우며, 새들은 노래를 시작하고, 사람들은 봄맞이 나들이와 축제들을 즐기죠. 

시적으로 봤을 때, 봄은 희망과 새 출발의 상징이기도 해요. 어둡고 차가운 겨울의 기억을 녹이고, 생명의 탄생과 재생의 기운을 품은 계절이기 때문에 많은 시인들은 봄을 사랑과 변화, 그리고 무한한 가능성의 은유로 표현하곤 하죠.

혹시 더 구체적으로 어떤 점이 궁금한지 알려주면 더 상세히 설명해줄 수 있어요!


In [69]:
# 역할 설정
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하고 짧게 대답해 주는 비서야.'},
        {'role':'user',   'content':'2020년 월드 시리즈는 누가 우승했어?'} # 질문
    ],
    temperature=1,
    frequency_penalty=0.5,
    # max_tokens=100
)
print(response.choices[0].message.content)

2020년 월드 시리즈는 로스앤젤레스 다저스가 우승했어요.


In [70]:
# 역할 설정(client는 이전 response를 몰라)
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하고 짧게 대답해 주는 비서야.'},
        {'role':'user',   'content':'그래서 몇대몇으로 어디에 이긴건데?'} # 질문
    ],
    temperature=1,
    frequency_penalty=0.5,
    # max_tokens=100
)
print(response.choices[0].message.content)

죄송합니다. 어떤 경기인지 알려주시면 정확히 도와드릴게요.


In [71]:
# 이전 답변을 포함하여 답변하기(few shot에도 사용하나 대화 히스토리용도를 훨씬 더 많이 씀)
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하게 답변해 주는 비서야'},
        {'role':'user',   'content':'2002년 월드컵에서 가장 화제가 되었던 나라는?'},
        {'role':'assistant','content':'예상을 뚫고 4강 진출한 한국입니다'},
        {'role':'user',    'content':'화제가 된 이유를 100단어 이내로 설명해 줘'}
    ],
    # max_tokens=100
)
print(response.choices[0].message.content)

2002년 한일 월드컵에서 한국이 4강에 진출한 것이 크게 화제가 되었습니다. 이는 아시아 국가 최초의 기록으로, 전 세계를 놀라게 했습니다. 선수들의 불굴의 투지와 열정, 그리고 한국 팬들의 열광적인 응원이 큰 힘이 되었으며, 특히 독일, 이탈리아, 스페인 등 강팀을 연속으로 이기면서 주목받았습니다. 또한, 대회 당시의 여러 논란과 함께, 한국 축구의 발전 가능성을 보여준 중요한 순간이었습니다. 이것이 바로 한국이 가장 화제가 된 이유입니다.


In [74]:
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system', 'content':'너는 친절하고 짧게 대답해 주는 비서야.'},
        {'role':'user',   'content':'2020년 월드 시리즈는 누가 우승했어?'},
        {'role':'assistant', 'content':'2020년 월드 시리즈는 로스앤젤레스 다저스가 우승했어요.'},
        {'role':'user',   'content':'그래서 몇대몇으로 어디에 이긴건데?'} # 질문
    ],
    temperature=1,
    frequency_penalty=0.5,
    # max_tokens=100
)
print(response.choices[0].message.content)

다저스는 탬파베이 레이스를 4대2로 이겼어요, 시애틀에서 열린 시리즈였어요.


In [79]:
# JSON형태로 output 받기(JSON형태의 문자로 받음)
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    response_format={'type':'json_object'}, # json형태로 응답(안쓰면 기본 text형태)
    messages=[
        {
            'role':'system', 
             'content':'너는 친절하고 짧게 대답해 주는 비서야. 답변은 반드시 JSON형태로 해줘'
        },
        {'role':'user',   'content':'2020년 월드 시리즈는 누가 우승했어?'} # 질문
    ],
    
    temperature=1,
    frequency_penalty=0.5,
)
print(response)

ChatCompletion(id='chatcmpl-ENsLjJ2YWD4v8AWiErNliR9WNiKB4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "winner": "로스앤젤레스 다저스",\n  "year": 2020\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789359171, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_1c77073857', usage=CompletionUsage(completion_tokens=25, prompt_tokens=53, total_tokens=78, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


In [81]:
result = response.choices[0].message.content
print(type(result), result)

<class 'str'> {
  "winner": "로스앤젤레스 다저스",
  "year": 2020
}


In [82]:
import json
dic_result = json.loads(result)
dic_result

{'winner': '로스앤젤레스 다저스', 'year': 2020}

In [3]:
# 웹 예제에서는 모두 함수화
# 답변을 잘 받기 위한 1. prompt > 2. 역할설정(system 메세지) 3. few shot(input token 사용多)
from dotenv import load_dotenv
from openai import OpenAI
def askGPT(prompt):
    'gpt-4.1-nano에게 prompt 요청 결과를 반환 : .env로드 -> clien객체 -> 요청 -> content만 반환'
    load_dotenv()
    client = OpenAI()
    response = client.chat.completions.create(
        model = 'gpt-4.1-nano',
        messages = [
            # {'role':'system', 'content':'당신은 텍스트를 잘 요약하는 전문 어시스턴트입니다'},
            {
                'role':'user',  
                 'content': f'''Your task is to summarize the text sentences in korean language.
Summarize in 2 lines. Use the format of a bullet point(✔️).
text : {prompt}'''
            }
        ]
    )
    return response.choices[0].message.content    

In [4]:
article = input('요약할 글을 입력하세요')
print(askGPT(article))

요약할 글을 입력하세요"오늘부터 휴대전화 요금이 25% 할인된다"는 가짜 정보가 요즘 시중에 돌고 있어 주의가 필요합니다.  SNS와 단체 대화방 등에는 "대통령 공약으로 휴대전화 기본요금이 인하돼 오늘부터 25% 할인된다", "직접 신청해야 할인받을 수 있다"는 메시지가 퍼지고 있는데요.  통신 3사의 실제 상담 전화번호까지 담겨 있어 이용자들의 혼란을 키우고 있습니다.  하지만 메시지에서 가리키는 25% 할인은 새로 적용된 혜택이 아닙니다.  이미 시행 중인 '선택약정 요금할인' 제도인데요.  휴대전화를 구매할 때 단말기 지원금을 받지 않은 이용자가 통신사와 약정을 맺으면 월정액 요금의 25%를 할인받을 수 있습니다.  통신 혜택 가짜 정보는 과거에도 여러 차례 반복해 확산됐었죠.  비슷한 할인 안내를 받았다면 그대로 믿기보다 통신사 공식 채널을 통해 사실 여부를 따져볼 필요가 있습니다.
- ✔️ 최근 SNS와 대화방에서 "25% 휴대전화 요금 할인"이라는 가짜 정보가 퍼지고 있으니 주의가 필요합니다.  
- ✔️ 이 할인 혜택은 이미 시행 중인 '선택약정 요금할인'으로, 통신사 공식 채널을 통해 확인하는 것이 중요합니다.


## 3. 스트리밍 응답 (Streaming)
기본적으로 OpenAI API는 요청에 대한 완료된 답변을 한꺼번에 반환합니다. 그러나 긴 답변의 경
우 스트리밍을 사용하면 마치 타이핑을 하듯이 토큰 단위로 차례로 응답을 받을 수 있습니다. 스
트리밍을 활용하면 사용자에게 실시간으로 응답을 표시하거나, 매우 긴 응답을 부분 부분 처리할
수 있습니다.
### 스트리밍이 필요한 경우
- 실시간 피드백: 사용자 경험을 개선하기 위해 답변 생성을 기다리는 동안 실시간으로 텍스트를
보여줄 때.
- 긴 응답 처리: 응답이 길어서 한꺼번에 받으면 메모리 사용이 많을 때, 토큰이 도착하는 대로
처리 가능.
- 중간 작업 가능: 응답을 받는 도중에도 다른 이벤트를 처리하거나 UI 업데이트를 할 수 있음.
### 스트리밍 사용 방법
OpenAI 파이썬 라이브러리에서 스트리밍을 사용하려면 요청 시 stream=True 옵션을 주면 됩니다
. 그러면 응답 객체 대신 **이터레이터(iterator)**를 반환하며, 이 이터레이터를 순회(for 문 등)하
면서 부분 응답(chunk)을 받을 수 있습니다.
다음은 스트리밍 응답을 처리하는 코드 예제입니다:

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()

In [16]:
# 스트리밍 예제 : 한글자씩 받아 출력
import time
messages = [
    {
        'role':'system',
        'content':'대한민국을 사랑하는 도우미입니다. 도시 이름 한글자씩 출력하는 도우미입니다. 다른 문장은 금지입니다'
    }, 
    {'role':'user', 'content':'아시아 도시명 5개를 알려줘요. 도시이름만 출력해 줘'}
]
response_stream = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
    stream=True
)
for chunk in response_stream:
    # 스트리밍으로 들어온 조작에서 추가된 content만 추출
    # print(chunk)
    chunk_message = chunk.choices[0].delta.content
    if chunk_message:
        print(chunk_message, end='/')
        time.sleep(0.5) # 0.5초

서울/
/베/이/징/
/도/쿄/
/뉴/델/리/
/방/콕/

위 코드를 실행하면 response_stream은 응답 스트림 객체가 되고, for 루프에서 순차적으로 응답조각을 받아옵니다. 각 chunk는 choices[0].delta에 현재 추가 생성된 텍스트 조각을 담고 있습니다 (완전한 메시지가 아니라 추가된 부분만을 담음). 이를 이어붙여 화면에 출력하면 모델이 답변을 조금씩 생성해가는 과정을 실시간으로 볼 수 있습니다. 예를 들어, 모델이 "안녕하세요, 만나서반갑습니다."라는 문장을 생성한다면, 스트리밍 출력은 사람이 타이핑하듯 안, 안녕, 안녕하세요, ...
차례로 출력될 것입니다. 스트리밍 모드는 주로 비동기 웹 애플리케이션이나 대화형 UI에서 활용되지만, Jupyter Notebook 환경에서도 위와 같이 동작 과정을 확인할 수 있습니다.

## 4. 시스템 메시지 활용
**시스템 메시지(system role message)**는 모델에게 전체 대화의 맥락이나 규칙을 알려주는 역할을 합니다. 시스템 메시지를 활용하면 AI의 말투, 행동 방식, 응답 형식 등을 조정할 수 있습니다. 시스템 메시지는 대화의 첫 번째 메시지로 넣는 경우가 많으며, 사용자에게는 보이지 않지만 모델에게는 강한 지침으로 작용합니다.

### 시스템 메시지의 역할
- 행동 지침: 모델이 따라야 할 규칙이나 목표를 제시 (예: "반말로 대답하지 마세요", "모든 응답에 이모티콘 하나를 포함하세요").
- 역할 부여: 모델에게 특정 인격이나 역할을 부여 (예: "너는 역사 전문가야", "너는 사용자를 돕는 비서야").
- 컨텍스트 설정: 대화 주제나 맥락을 사전에 설정 (예: "이 대화는 의료 상담입니다", "사용자는 프로그래밍 도움을 요청할 것입니다"). 시스템 메시지는 한 번 설정하면 해당 대화 내내 지속적으로 모델의 응답 스타일에 영향을 미치지만, 필요한 경우 대화 중간에 새로운 시스템 메시지를 추가하여 조정할 수도 있습니다 (예를 들어, 새로운 규칙을 추가).

### 시스템 메시지 사용 예제
시스템 메시지를 사용하여 모델의 말투를 바꿔보겠습니다. 모델에게 "해적처럼 말하는 코딩 도우미"라는 캐릭터를 부여한 후, 사용자의 질문에 답하게 해보겠습니다

In [18]:
messages = [
    {'role':'system', 'content':'You are a coding assistant that talks like a pirate.'},
    {'role':'user', 'content':'Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게 하는지 한국어로 대답해 줘'}
]
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages
)
print(response.choices[0].message.content)

아이고, 내 사랑! 파이썬에서 어떤 객체가 특정 클래스의 인스턴스인지 확인하려면요, `isinstance()`라는 함수를 써요. 예를 들어서 말이야, 만약 `obj`라는 객체가 `Dog`라는 클래스의 인스턴스인지 알고 싶으면 이렇게 하시면 된다구요:

```python
isinstance(obj, Dog)
```

이 함수는 맞으면 `True`, 아니면 `False`를 돌려준답니다. 알겠어요? 너무 어렵지 않지요?


위 예제의 시스템 메시지는 영어로 작성되었지만(물론 한국어로 지시해도 됩니다), "당신은 해적처
럼 말하는 코딩 도우미"라는 지침을 줍니다. 그 다음 사용자 질문은 일반적으로 "Python에서 객
체가 특정 클래스의 인스턴스인지 어떻게 확인하나요?"라는 내용입니다. 시스템 메시지 덕분에,
모델의 답변은 아마도 해적 말투로 나올 것입니다.
이처럼 동일한 질문이라도 시스템 메시지를 통해 모델의 답변 스타일이나 관점을 크게 바꿀 수
있습니다. 필요에 따라 시스템 메시지를 활용하여 프로젝트의 톤앤매너에 맞는 응답을 얻도록 조
정하세요.

> 참고: 시스템 메시지는 사용자가 직접 볼 수 없으므로, 중요한 지시사항(예: "사용자에게 욕설을
하지 마라")은 반드시 시스템 메시지로 전달해야 합니다. 모델은 사용자 메시지의 내용보다 시스
템 메시지의 지시에 우선순위를 두도록 설계되어 있습니다.

## 5. 고급 활용법
이 섹션에서는 Chat Completions API를 보다 효율적으로 사용하기 위한 고급 기법들을 다룹니다.
토큰 사용을 최적화하여 비용을 절감하는 방법과, API 호출 시 발생할 수 있는 오류를 처리하는
방법을 설명합니다.

### 토큰 최적화 및 비용 절감
OpenAI API 비용은 사용한 토큰(token) 수에 비례하여 청구됩니다. 따라서 동일한 작업을 하더라
도 토큰을 적게 사용하면 비용이 줄어들고, 응답 속도도 빨라집니다. GPT-4o 모델은 최대 128k
토큰의 컨텍스트를 지원하지만, 불필요하게 많은 토큰을 사용하지 않도록 최적화하는 것이 중요
합니다.

토큰 최적화를 위한 팁:
- 짧고 명확한 프롬프트: 시스템 메시지와 사용자 메시지를 불필요하게 장황하게 쓰지 않고 간결하게 작성합니다. 예를 들어 동일한 지시라도 "간결하게 답변해주세요."는 "부디 당신의 답변을 최대한 간략하게 제공해 주셨으면 합니다."보다 적은 토큰으로 같은 의미를 전달합니다.

- 대화 내역 관리: 이전 대화 기록을 얼마나 포함시킬지 결정해야 합니다. 모든 이전 메시지를 매번 보낼 필요는 없습니다. 중요한 맥락만 남기고 요약하거나 일부 생략하여 토큰을 줄입니다.

- 모델 선택: 반드시 GPT-4o 수준의 성능이 필요하지 않은 작업에는 GPT-4o-mini와 같은 더 작은 모델을 사용해 비용을 절감할 수 있습니다. (GPT-4o-mini는 GPT-4o보다 비용이 훨씬 저렴하여 일상적인 작업에 적합합니다.)

- max_tokens 파라미터 활용: 응답의 최대 길이를 설정하여 너무 긴 답변이 나오지 않도록 제어합니다. 예를 들어 요약 생성 등의 작업에서는 max_tokens를 짧게 설정해 모델이 알아서 간결한 답을 내놓게 유도할 수 있습니다.
스트리밍과 부분 처리: 앞서 소개한 스트리밍 기능을 사용하면, 매우 긴 응답의 경우 중간 중간 출력 결과를 확인하며 필요에 따라 조기에 중단하는 등의 대응을 할 수 있습니다. 추가로, OpenAI는 Batch API 등을 통해 다수의 요청을 한 번에 보내 비용을 절약하는 방법을 제공하기도 합니다. 다만 이 튜토리얼의 범위를 벗어나므로 자세한 내용은 OpenAI 공식 문서를 참
고하세요.
토큰 최적화의 효과를 확인하고 싶다면, 응답 객체의 usage 정보를 출력해볼 수 있습니다. response.usage에는 이번 요청에서 사용된 prompt_tokens(입력 토큰 수), completion_tokens(출력토큰 수), total_tokens(합계)가 담겨 있습니다. 예를 들어:

In [20]:
response.usage # 입력 토큰 수 :completion_tokens, 출력토큰수:prompt_tokens

CompletionUsage(completion_tokens=117, prompt_tokens=48, total_tokens=165, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

In [22]:
response.usage.completion_tokens, response.usage.prompt_tokens

(117, 48)

이런 정보를 토대로 모델이 과도하게 긴 답변을 내놓지는 않았는지 모니터링하고, 프롬프트를 조정하는 피드백 loop을 거치면 점점 효율적으로 API를 활용할 수 있습니다.

### 에러 핸들링 및 예외 처리
OpenAI API를 사용하는 애플리케이션을 개발할 때는 각종 오류 상황을 대비해야 합니다. 주로 발
생할 수 있는 예외 상황과 대처 방안은 다음과 같습니다:

- 네트워크 오류 또는 타임아웃: 인터넷 연결 문제나 일시적인 서버 응답 지연으로 요청이 실패할 수 있습니다. 이 경우 요청을 재시도하거나, 백엔드에서 지수적 지연 전략(exponential backoff )을 사용해 일정 시간 후 다시 시도하는 것이 좋습니다.
- 레이트 리미트 (Rate Limit) 초과: OpenAI API는 일정 기간당 요청 허용량을 초과하면 RateLimitError를 발생시킵니다. 이 경우 일정 시간 대기 후 재시도하거나, 요청 빈도를 낮추는 조정이 필요합니다.
7
- 유효하지 않은 요청: 모델 이름 오타, 매개변수 형식 오류 등으로 InvalidRequestError가 발생할수 있습니다. 이런 오류는 API 호출 전에 코드에서 철저한 검증을 통해 예방하는 것이 좋습니다.(Dale쓸 때 이미지 처리시 InvalidRequestError생길수 있음)
- API 키 오류: 잘못된 API 키나 권한 문제로 인증 오류(AuthenticationError)가 발생할 수 있으므로, API 키가 정확하고 유효한지 확인해야 합니다. 파이썬 라이브러리를 사용할 때 이러한 오류들은 openai.error 모듈 내 예외 클래스로 나타납니다. 일반적인 최상위 예외는 openai.error.OpenAIError이며, 모든 OpenAI 관련 예외의 부모 클래스입니다. 간단한 예외 처리 예제를 보겠습니다:


In [25]:
import openai
try:
    res = client.chat.completions.create(
        model='gpt-4.1-nano',
        messages=[{'role':'user', 'content':'에러를 일으켜 보아요'}],
        timeout=0.01 # 답변을 0.01초만에 받겠다
    )
# except openai.APITimeoutError 모든 openai 에러는 OpenAIError로부터 상속받음
except openai.OpenAIError as e: 
    print('시간을 초과하였습니다.')
    print(e)
    print(type(e))

시간을 초과하였습니다.
Request timed out.
<class 'openai.APITimeoutError'>


In [26]:
res

NameError: name 'res' is not defined

위 코드에서 timeout=5는 응답이 5초 안에 없으면 OpenAIError를 발생시키도록 한 것으로, 강제로 타임아웃 상황을 연출하기 위한 예시입니다. RateLimitError는 별도로 캐치하여 사용자에게 요청 제한 메세지를 보여주고, 그 외 모든 OpenAI 오류는 일반적으로 메시지(e)를 출력하도록 했습니다. 실제 애플리케이션에서는 오류 종류에 따라 로깅을 남기고, 필요하면 재시도 로직을 넣는 등 더 정교한 대응을 구현할 수 있습니다. 

마지막으로, 예상하지 못한 예외 상황(예: JSON 디코딩 오류나 타입 오류 등)이 발생할 수 있으므로, API 호출 코드 주위에는 일반 예외 처리도 넣어서 프로그램이 갑자기 중단되지 않도록 만드는것이 좋습니다.

## 6. 실전 프로젝트 예제
마지막으로, 앞서 배운 내용을 종합하여 실제 응용 사례로 여러 번의 대화가 오가는 챗봇 구현를 간단히 살펴보겠습니다.

### 간단한 대화형 챗봇 구현
OpenAI Chat Completions API를 사용하면 비교적 적은 코드로 대화형 챗봇을 만들 수 있습니다. 여기서는 콘솔에서 사용자의 입력을 받아 모델의 응답을 출력하는 간단한 챗봇을 구현해봅니다. 이 챗봇은 이전 대화 맥락을 기억하여 연속적인 대화를 주고받을 수 있습니다.

In [38]:
# 대화 이력을 저장할 list
chat_history = [
    {'role':'system', 'content':'당신은 유능한 AI 상담원입니다.'},
]
print('쳇봇 시작(종료 : exit, quit, bye, 종료)')
input_tokens = 0
output_tokens = 0
while True:
    user_input = input('사용자 질문:').strip()
    if user_input.lower() in ['exit', 'quit', 'bye', '종료']:
        print('쳇봇 종료')
        break
    if user_input.strip() == '':
        continue
    # 사용자 질문(user_input)을 chat_history에 append
    chat_history.append(
        {'role':'user', 'content':user_input}
    )
    # 답변 출력 & chat_history에 assistant로 append
    try:
        # openai API 호출
        response = client.chat.completions.create(
            model = 'gpt-4.1-nano',
            messages=chat_history
        )
        input_tokens += response.usage.completion_tokens # 요청의 입력토큰수 누적
        output_tokens += response.usage.prompt_tokens # 요청의 출력토큰 수 누적
        
    except openai.OpenAIError as e:
        print('오류가 발생하였습니다. admin에게 요청해 주세요')
        break
        
    assistant_reply = response.choices[0].message.content.strip()
    print('AI 답변 :', assistant_reply)
    # chat_history에 너무 많은 질문과 답변이 쌓이면 앞의 일부를 요약하는 작업
    chat_history.append(
        {'role':'assistant', 'content':assistant_reply}
    )
print('소요한 입력토큰 :', input_tokens)
print('소요한 출력토큰 :', output_tokens)
print('소요된 비용 :', ((input_tokens+output_tokens*4)/1000000)*0.1 , '$')

쳇봇 시작(종료 : exit, quit, bye, 종료)
사용자 질문:2006년생은 음주 가능해?
AI 답변 : 한국에서는 만 19세부터 음주가 법적으로 허용됩니다. 따라서 2006년생이라면 2025년에 만 19세가 되므로, 그때부터 음주가 가능해집니다. 현재는 만 17세이기 때문에 음주가 불법입니다.
사용자 질문:그들은 주로 어떤 주종을 좋아할까?
AI 답변 : 2006년생들이 좋아하는 주종은 개인의 취향과 트렌드에 따라 다를 수 있지만, 일반적으로 젊은 세대는 다음과 같은 주종을 즐기는 경우가 많습니다:

1. 맥주: 다양한 맛과 브랜드가 있어 인기가 높고, 맥주에 대한 관심도 높습니다.
2. 칵테일 및 믹서음료: 과일 맛이 나는 음료와 간단한 칵테일은 인기 있는 선택지입니다.
3. 무알코올 음료: 미성년자들은 법적으로 술을 마실 수 없기 때문에, 무알코올 맥주, 모조 칵테일, 에이드류 등을 선호합니다.
4. 전통주 및 소주: 한국의 전통주나 소주는 성인들이 주로 즐기지만, 일부는 관심을 갖고 맛보기도 합니다.

중요한 점은, 미성년자는 법적으로 알코올 섭취가 금지되어 있으며, 건강과 법적 문제를 고려해야 한다는 것입니다.
사용자 질문:exit
쳇봇 종료
소요한 입력토큰 : 293
소요한 출력토큰 : 147
소요된 비용 : 8.81e-05 $


In [35]:
print('입력토큰수 :', response.usage.completion_tokens)
print('출력토큰수 :', response.usage.prompt_tokens)

입력토큰수 : 159
출력토큰수 : 285


위 코드는 while 루프를 돌면서 사용자 입력을 받습니다. "종료"라고 입력하면 루프를 빠져나와 챗
봇이 종료됩니다. 각 반복에서 사용자의 입력을 chat_history에 추가한 후, 해당 chat_history를 그
대로 모델에게 보내 응답을 받습니다. 응답을 출력하고, 다시 chat_history에 추가하여 맥락을 유
지합니다. 시스템 메시지로 초반에 상담원으로서의 태도를 지정했기 때문에, AI는 공손하고 상세한
답변을 지속적으로 생성할 것입니다.
이처럼 간단한 구조만으로도 사용자와 지속적인 맥락을 가진 대화를 주고받는 챗봇을 만들 수 있
습니다. 실제 응용에서는 여기에 GUI를 입히거나, 웹 서비스와 연결하거나, 데이터베이스와 연동
하는 등의 확장이 가능하지만, 핵심 로직은 위와 같습니다.